In [1]:
import numpy as np
import awkward as ak
import matplotlib.pyplot as plt
from coffea.nanoevents import NanoEventsFactory, NanoAODSchema

In [2]:
file_TTtoLNu2Q = "root://xrootd-cms.infn.it///store/mc/RunIII2024Summer24NanoAODv15/TTtoLNu2Q_TuneCP5_13p6TeV_powheg-pythia8/NANOAODSIM/150X_mcRun3_2024_realistic_v2-v2/2520000/bbd9a56d-d95d-44db-89cc-8ba93ac4b7cd.root"
file_TTG1Jets =  "root://xrootd-cms.infn.it///store/mc/RunIII2024Summer24NanoAODv15/TTG-1Jets_TuneCP5_13p6TeV_amcatnloFXFXold-pythia8/NANOAODSIM/150X_mcRun3_2024_realistic_v2-v2/120000/b3034ff0-176a-4a0d-8d44-fb1002ba0436.root"

factory_TTtoLNu2Q = NanoEventsFactory.from_root(
    f"{file_TTtoLNu2Q}:Events",
    schemaclass=NanoAODSchema,
)
events_TTtoLNu2Q = factory_TTtoLNu2Q.events()

factory_TTG1Jets = NanoEventsFactory.from_root(
    f"{file_TTG1Jets}:Events",
    schemaclass=NanoAODSchema,
)
events_TTG1Jets = factory_TTG1Jets.events()

In [3]:
events_TTtoLNu2Q_pileup = events_TTtoLNu2Q.Pileup.nPU
events_TTG1Jets_pileup = events_TTG1Jets.Pileup.nPU

In [10]:
def draw_statbox_manual(hist, x1, y1, x2, y2, color, label=None):

    stats = ROOT.TPaveText(x1, y1, x2, y2, "NDC")
    stats.SetFillColor(0)
    stats.SetBorderSize(1)
    stats.SetTextColor(color)
    stats.SetTextFont(42)
    stats.SetTextSize(0.02)

    nbins = hist.GetNbinsX()

    underflow = hist.GetBinContent(0)
    overflow  = hist.GetBinContent(nbins + 1)
    integral  = hist.Integral(1, nbins)

    if label is not None:
        stats.AddText(label)

    stats.AddText(f"Entries   = {int(hist.GetEntries())}")
    stats.AddText(f"Mean      = {hist.GetMean():.2f}")
    stats.AddText(f"RMS       = {hist.GetRMS():.2f}")
    stats.AddText(f"Underflow = {underflow:.2f}")
    stats.AddText(f"Overflow  = {overflow:.2f}")
    stats.AddText(f"Integral  = {integral:.2f}")

    stats.Draw()

    return stats

In [11]:
import ROOT 

ROOT.gStyle.SetOptStat(1110)

c = ROOT.TCanvas()

# First histogram
h_tt.Draw()
c.Update()

# Second histogram (temporarily by itself)
h_ttg.Draw()
c.Update()

# Now redraw both
c.Clear()
h_tt.Draw("HIST")
h_ttg.Draw("HIST SAME")
c.Update()

# Legend (move it away from the stat boxes)
leg = ROOT.TLegend(0.15, 0.75, 0.38, 0.88)
leg.SetBorderSize(0)
leg.SetFillStyle(0)
leg.AddEntry(h_tt, "TTtoLNu2Q", "l")
leg.AddEntry(h_ttg, "TTG1Jets", "l")
leg.Draw()

# Manual stat boxes
stat_tt = draw_statbox_manual(
    h_tt,
    0.72, 0.80, 0.95, 0.95,
    ROOT.kBlue,
    "TTtoLNu2Q"
)

stat_ttg = draw_statbox_manual(
    h_ttg,
    0.72, 0.58, 0.95, 0.78,
    ROOT.kRed,
    "TTG1Jets"
)

c.Modified()
c.Update()

c.Modified()
c.Update()

c.SaveAs("pileup_comparison.pdf")
c.SaveAs("pileup_comparison.png")

Info in <TCanvas::Print>: pdf file pileup_comparison.pdf has been created
Info in <TCanvas::Print>: png file pileup_comparison.png has been created
